# Advanced AI - Assignment 1: k-Nearest Neighbors

Implement the missing pieces in `advanced_ai/classifiers/k_nearest_neighbor.py`, then use the cells below to check your work.

Provided for you:
- dataset loading and splitting
- metrics and task runners
- validation / evaluation loops
- submission packaging

Do not rebuild from scratch:
- project structure
- metric functions
- repeated experiment loops
- submission zipping


## Core Equations And Rules

Use these definitions consistently in your implementation.

- Euclidean distance:

$$
d_{L_2}(x, z) = \sqrt{\sum_j (x_j - z_j)^2}
$$

- Manhattan distance:

$$
d_{L_1}(x, z) = \sum_j |x_j - z_j|
$$

- Cosine similarity and cosine distance:

$$
\cos(x, z) = \frac{x^\top z}{\|x\|_2 \|z\|_2}, \qquad d_{\cos}(x, z) = 1 - \cos(x, z)
$$

- Uniform classification vote:

$$
\hat{y} = \arg\max_c \sum_{i \in N_k(x)} \mathbf{1}[y_i = c]
$$

- Distance-weighted classification:

$$
w_i = \frac{1}{d(x, x_i) + 10^{-12}}, \qquad \hat{y} = \arg\max_c \sum_{i \in N_k(x)} w_i\,\mathbf{1}[y_i = c]
$$

- Uniform regression:

$$
\hat{y} = \frac{1}{k} \sum_{i \in N_k(x)} y_i
$$

- Distance-weighted regression:

$$
\hat{y} = \frac{\sum_{i \in N_k(x)} w_i y_i}{\sum_{i \in N_k(x)} w_i}
$$

- Standardization with training-set statistics only:

$$
x'_j = \frac{x_j - \mu_j}{\sigma_j}
$$

Implementation expectations:
- the `two_loops`, `one_loop`, and `no_loops` distance functions should agree numerically
- the no-loop version should use vectorized NumPy operations
- ties in classification should be broken by choosing the smaller class label



## Data Files And Preparation

Expected prepared files:
- `datasets/banknote_authentication.csv`
- `datasets/california_housing.npz`
- `datasets/text_retrieval_embeddings.npz`
- `datasets/network_anomaly.npz`

Expected dataset formats:
- `banknote_authentication.csv`: comma-separated numeric table, last column is the label
- `california_housing.npz`: keys `X`, `y`
- `text_retrieval_embeddings.npz`: keys `query_embeddings`, `doc_embeddings`, `relevance`, `query_lengths`, `doc_lengths`; metadata such as `query_ids`, `doc_ids`, `model_name`, `dataset_name` may also be present
- `network_anomaly.npz`: keys `X_train`, `X_val`, `y_val`, `X_test`, `y_test`

If you need to create the prepared artifacts from scratch, run:

```bash
bash scripts/prepare_all_data.sh
```

Useful variants:

```bash
bash scripts/prepare_all_data.sh --core-only
bash scripts/prepare_all_data.sh --skip-retrieval
bash scripts/prepare_all_data.sh --overwrite
bash scripts/prepare_all_data.sh --skip-install
```

Verification scripts:

```bash
bash scripts/check_prepare_datasets.sh
bash scripts/check_prepare_embeddings.sh
bash scripts/check_prepare_all_data.sh
```

Notes:
- the shell wrapper creates and uses the conda environment `advanced-ai-assignment1-knn-data`
- retrieval preparation downloads SciFact and encodes it with `sentence-transformers/all-MiniLM-L6-v2`
- anomaly preparation downloads KDD Cup 99, encodes categorical features, and scales numeric features
- `--skip-install` should be used only when the dedicated conda environment already has the required dependencies installed
- the check scripts reject incompatible flags such as `--core-only`, `--skip-retrieval`, and `--skip-anomaly` when those flags would make a success message misleading


## Workflow Summary

1. Read the TODO blocks.
2. Fill in `advanced_ai/classifiers/k_nearest_neighbor.py`.
3. Run `python -m advanced_ai.self_checks` until the checks pass.
4. Run the required tasks:

```bash
python banknote_classification.py
python california_regression.py
```

5. If you finish early, try at most one optional extension:

```bash
python text_retrieval_optional.py
python network_anomaly_optional.py
```


In [ ]:
from __future__ import annotations

import numpy as np

from advanced_ai.classifiers.k_nearest_neighbor import KNearestNeighbor
from advanced_ai.self_checks import main as run_self_checks

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    return np.max(np.abs(x - y) / np.maximum(1e-8, np.abs(x) + np.abs(y)))


## TODO: Edit the k-NN core

Open `advanced_ai/classifiers/k_nearest_neighbor.py` and fill in the missing methods.

Leave the project structure, metrics, and experiment loops unchanged.


## TODO 1: Self-Checks

Run the self-check suite first to confirm your environment is healthy and outputs are reproducible.


In [ ]:
run_self_checks()


## TODO 2: Distance Checks

After implementing the distance functions, the three implementations should agree.


In [ ]:
np.random.seed(231)
X_train = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 2.0]])
X_test = np.array([[1.0, 1.0], [2.0, 0.0]])

knn = KNearestNeighbor()
knn.fit(X_train, np.array([0, 1, 1]))

d2 = knn.compute_distances_two_loops(X_test, metric="l2")
d1 = knn.compute_distances_one_loop(X_test, metric="l2")
d0 = knn.compute_distances_no_loops(X_test, metric="l2")

print("two vs one:", rel_error(d2, d1))
print("two vs zero:", rel_error(d2, d0))


## TODO 3: Prediction Checks

Once neighbor search works, confirm that classification and regression predictions run end-to-end.


In [ ]:
X_train = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [2.0, 2.0]])
X_test = np.array([[0.9, 0.1], [0.2, 0.9]])

knn = KNearestNeighbor()
knn.fit(X_train, np.array([0, 0, 1, 1]))
print("classification:", knn.predict(X_test, k=3, metric="l2", task="classification"))

knn.fit(X_train, np.array([0.0, 0.0, 1.0, 4.0]))
print("regression:", knn.predict(X_test, k=2, metric="l2", task="regression"))


## TODO 4: Required Tasks

These scripts use the provided assignment scaffold so you can focus on the algorithm rather than boilerplate.

Expected prepared datasets:
- `datasets/banknote_authentication.csv`
- `datasets/california_housing.npz`


In [ ]:
%run banknote_classification.py


In [ ]:
%run california_regression.py


## TODO A: Optional Retrieval Extension

Use this only if the instructor has provided `datasets/text_retrieval_embeddings.npz`.


In [ ]:
# %run text_retrieval_optional.py


## TODO B: Optional Anomaly Extension

Use this only if the instructor has provided `datasets/network_anomaly.npz`.


In [ ]:
# %run network_anomaly_optional.py


## TODO: Inline Questions

1. Why can `k=1` overfit while large `k` can underfit?
2. When would cosine similarity be preferred over Euclidean distance?
3. Why must scaling statistics come from the training set only?
4. When does brute-force k-NN become too expensive in production?

Write your answers in your report or directly below in new markdown cells.
